# Two Agents in Conversation [Agent Patterns - Module 13]

> **MLCourse - Agentic AI - Agent Patterns**

AutoGen's central abstraction is the **team**: several agents sharing one
message stream, taking turns, until a **termination condition** fires.

This is a genuinely different mental model from the frameworks you have used:

- **LangGraph** - you draw the control flow as a graph. Edges are explicit.
- **CrewAI** - you assign roles and a process; the framework sequences tasks.
- **AutoGen** - you put agents in a room and specify when to stop talking.

That last one is powerful and it is also the risk. A conversation with no
good stopping rule is an infinite loop with a token meter attached, which is
why termination gets its own section here.

### What you will learn

1. `RoundRobinGroupChat` and how turn-taking works.
2. Termination conditions, and composing them with `|`.
3. Watching a team converse with `Console`.
4. Reading the conversation transcript and its token cost.
5. Why every team needs a hard turn limit, always.

### Key takeaways

- Termination is a first-class design decision, not an afterthought.
- Always AND/OR a `MaxMessageTermination` in. Always.
- The transcript is the artifact - inspect it, do not just take the last line.

### Setup: imports, environment, track discovery


In [ ]:
import os
import sys
import json
import time
import random
import asyncio
from pathlib import Path
from dotenv import load_dotenv

def _find_track(depth=6):
    p = Path.cwd()
    for _ in range(depth):
        if (p / "03_agentic_ai").is_dir():
            return p
        p = p.parent
    return Path.cwd()

TRACK = _find_track()
# The .env lives INSIDE the track folder, not the repo root that
# _find_track() returns - joining ".env" onto TRACK alone is a silent no-op.
load_dotenv(TRACK / "03_agentic_ai" / ".env", override=False)

GROQ_API_KEY = os.environ["GROQ_API_KEY"]   # loud failure if missing, by design
MODEL = "qwen/qwen3.8-27b"                   # Groq-hosted; never OpenAI

print(f"Track root : {TRACK}")
print(f"Model      : {MODEL} (via Groq's OpenAI-compatible endpoint)")


### Point AutoGen at Groq


In [ ]:
# AutoGen ships an OpenAI client. Groq exposes an OpenAI-COMPATIBLE endpoint,
# so we reuse that client and only change the base_url. This is the standard
# way to run AutoGen on a non-OpenAI provider - there is no Groq-specific
# client to install.
#
# `model_info` is REQUIRED for any model AutoGen does not have a built-in
# capability table for. Without it you get a ValueError before a single
# request goes out. You are telling the framework what the model can do.

from autogen_ext.models.openai import OpenAIChatCompletionClient

def make_client():
    return OpenAIChatCompletionClient(
        model=MODEL,
        api_key=GROQ_API_KEY,
        base_url="https://api.groq.com/openai/v1",   # <- the only Groq-specific line
        temperature=0.0,
        max_tokens=500,                              # free tier is 8000 TPM
        model_info={
            "vision": False,
            "function_calling": True,
            "json_output": True,
            "family": "unknown",
            "structured_output": False,
        },
    )

print("make_client() ready")


### The shared task


In [ ]:
# This module solves ONE task in three frameworks so the comparison is about
# the frameworks, not the problem. It is the same shape as the CrewAI crew in
# 04_crewai/01_fundamentals/05_research_assistant_crew: a writer produces a
# short piece from fixed reference notes, a critic reviews it, the writer
# revises. Deliberately tiny - we are studying plumbing, not prose.

NOTES = """Research notes: agent frameworks, 2026.
- Frameworks matured: CrewAI (role-based crews), LangGraph (explicit graphs),
  AutoGen (conversational agents).
- Agents are moving from demos to production.
- Main challenges: reliability, cost control, observability."""

TASK = ("Using ONLY the notes below, write a 2-sentence summary for an "
        "engineering newsletter.\n\n" + NOTES)

print(TASK)


### 1. Writer and critic

Two agents. The writer drafts; the critic reviews. The critic has one
special instruction: emit a specific word when satisfied. That word is what
the termination condition watches for.

This "magic word" protocol feels crude, and it is - but it is explicit, it is
in plain sight, and you can grep the transcript for it. Compare that to a
framework where the stopping rule is buried in the engine.

### The two agents


In [ ]:
from autogen_agentchat.agents import AssistantAgent

client = make_client()

writer = AssistantAgent(
    name="writer",
    model_client=client,
    system_message=("You write concise engineering newsletter copy from the "
                    "facts you are given. No bullet points, no headings. "
                    "When the critic gives feedback, produce a revised "
                    "version - the full text, not a description of changes."),
)

critic = AssistantAgent(
    name="critic",
    model_client=client,
    system_message=("You review newsletter copy for accuracy against the "
                    "source notes and for concision. Give AT MOST one "
                    "specific, actionable improvement in one sentence. "
                    "If the draft is accurate and under 3 sentences, reply "
                    "with exactly: APPROVED"),
)

print("agents:", writer.name, "+", critic.name)


### 2. Termination conditions

A `TerminationCondition` is checked after every message. AutoGen ships
several, and they compose with `|` (or) and `&` (and):

| Condition | Fires when |
|---|---|
| `TextMentionTermination("APPROVED")` | that string appears in a message |
| `MaxMessageTermination(n)` | n messages have been produced |
| `TokenUsageTermination(...)` | a token budget is exhausted |
| `TimeoutTermination(seconds)` | wall-clock limit |
| `ExternalTermination()` | your code calls `.set()` |

**The rule that matters:** the semantic condition (`APPROVED`) says when
you *want* to stop; the mechanical one (`MaxMessageTermination`) guarantees
you *will*. A model that never says the magic word - because it is being
polite, or restating the word inside a sentence, or having an off day - turns
your team into a spend loop. Compose both, every time.

### Compose the stopping rule


In [ ]:
from autogen_agentchat.conditions import TextMentionTermination, MaxMessageTermination
from autogen_agentchat.teams import RoundRobinGroupChat

approved = TextMentionTermination("APPROVED")   # what we want
safety = MaxMessageTermination(7)               # what we guarantee

team = RoundRobinGroupChat(
    participants=[writer, critic],   # turn order is exactly this list
    termination_condition=approved | safety,
)

print("termination:", type(approved).__name__, "OR", type(safety).__name__)
print("max messages:", 7)


`RoundRobinGroupChat` is the simplest team: participants speak strictly in
list order. AutoGen also has `SelectorGroupChat`, where a model picks the
next speaker, and `Swarm`, where agents hand off explicitly.

Start with round-robin. It is deterministic, cheap, and easy to reason about.
Reach for a selector only when you can articulate why fixed order fails - a
model-chosen speaker adds an LLM call per turn *and* a new failure mode.

### 3. Run the conversation


### Watch it live


In [ ]:
# Console() consumes the stream and prints each message as it arrives.

from autogen_agentchat.ui import Console

t0 = time.time()
result = await Console(team.run_stream(task=TASK))
elapsed = time.time() - t0

print(f"\nstop_reason: {result.stop_reason}")
print(f"elapsed    : {elapsed:.1f}s")


### The transcript, and what it cost


In [ ]:
total_in = total_out = 0
print(f"{'#':>2}  {'source':10s}{'in':>7s}{'out':>7s}  text")
print("-" * 78)
for i, m in enumerate(result.messages):
    u = getattr(m, "models_usage", None)
    pin = u.prompt_tokens if u else 0
    pout = u.completion_tokens if u else 0
    total_in += pin
    total_out += pout
    print(f"{i:>2}  {getattr(m,'source','-'):10s}{pin:>7d}{pout:>7d}  "
          f"{str(getattr(m,'content',''))[:44]}")
print("-" * 78)
print(f"{'':2}  {'TOTAL':10s}{total_in:>7d}{total_out:>7d}")
print()
print(f"{len(result.messages)} messages cost {total_in + total_out} tokens.")
print("Note how prompt tokens GROW each turn: every agent re-reads the whole")
print("conversation. That is the hidden cost of the conversational model, and")
print("it is quadratic in turns.")


### The quadratic cost of conversation

Look at the `in` column. Each turn re-sends the whole transcript, so a
conversation of *n* turns costs roughly *n²/2* messages' worth of prompt
tokens. On a free tier that is the difference between a demo that runs and
one that 429s halfway.

Mitigations, in order of how often you will actually need them:

1. **Fewer turns.** A tight termination condition is a cost control.
2. **Shorter messages.** `max_tokens` on the client, and system prompts that
   demand brevity.
3. **Summarise the history** when it grows past a threshold - see
   `06_agent_patterns/02_memory_at_scale`.

### Did the critic actually approve, or did we hit the ceiling?


In [ ]:
hit_limit = "Maximum number of messages" in str(result.stop_reason)
approved_text = any("APPROVED" in str(getattr(m, "content", ""))
                    for m in result.messages)

print("stop_reason  :", result.stop_reason)
print("critic said APPROVED:", approved_text)
print("hit the safety limit:", hit_limit)
print()
if hit_limit:
    print("The safety net did its job. In production this should be an ALERT,")
    print("not a silent success - the team did not converge, it was cut off.")
else:
    print("Converged on the semantic condition, as designed.")


### Clean up


In [ ]:
await client.close()
print("client closed")


### Pitfalls recap

- **No mechanical termination.** A team with only `TextMentionTermination`
  can run forever. Always OR in `MaxMessageTermination`.
- **Treating "hit the limit" as success.** Check `stop_reason`. Non-convergence
  is a real outcome that deserves an alert.
- **The magic word appearing by accident.** "I would not say APPROVED yet"
  terminates the chat. Pick a token unlikely to occur in prose, and instruct
  the agent to emit it alone.
- **Ignoring the quadratic prompt growth.** It is the dominant cost of
  conversational frameworks.
- **Reaching for `SelectorGroupChat` too early.** It costs an extra LLM call
  per turn and makes runs non-deterministic.

### Next

Notebook 03 gives the agents a tool, and shows how AutoGen's tool loop
differs from the ones you have already used.